# 02 — Pipeline Demo

Runs the full gap-scoring pipeline and visualises the results.

**Pre-requisites:**
1. `python scripts/download_data.py` completed
2. `python -c "from src.ingestion.loaders import build_master_dataset; build_master_dataset()"` completed
3. `ANTHROPIC_API_KEY` set in `.env` (needed only for the LLM query cells)

In [ ]:
import sys, os
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import geopandas as gpd
from dotenv import load_dotenv
load_dotenv('../.env')

from src.scoring.gap_score import compute_gap_scores, rank_crises

print('Ready.')

---
## 1. Load Master Dataset

In [ ]:
master = pd.read_parquet('../data/processed/master.parquet')
print(f'Shape: {master.shape}')
print(f'Years: {master["year"].min()}–{master["year"].max()}')
print(f'Countries: {master["country_iso3"].nunique()}')
master.head()

In [ ]:
# Filter to rows useful for gap analysis:
# - have both PIN (need signal) and funding data
# - sensible year range
analysis = master[
    master['pin'].notna() &
    master['requirements_usd'].notna() &
    master['year'].between(2020, 2026)
].copy()

print(f'Analysis-ready rows: {len(analysis)}')
analysis.head()

---
## 2. Score Every Row

In [ ]:
scored = compute_gap_scores(analysis)
print('Added columns:', [c for c in scored.columns if c not in analysis.columns])
scored[['country_iso3', 'year', 'sector', 'pin', 'coverage_ratio', 'coverage_gap', 'need_scale', 'gap_score', 'in_scope']].head(10)

In [ ]:
# Gap score distribution
fig = px.histogram(
    scored[scored['in_scope']], x='gap_score', nbins=40,
    title='Gap Score Distribution (in-scope rows only)',
    labels={'gap_score': 'Gap Score'},
    color_discrete_sequence=['#e63946']
)
fig.show()

---
## 3. Rank — Top 20 Most Overlooked Crises

In [ ]:
ranked = rank_crises(scored, top_n=20)
display_cols = ['rank', 'country_iso3', 'year', 'sector', 'pin',
                'requirements_usd', 'funding_usd', 'coverage_ratio', 'gap_score']
ranked[display_cols]

In [ ]:
# Bar chart: gap score with coverage ratio colour
ranked['label'] = ranked['country_iso3'] + ' ' + ranked['year'].astype(str)

fig = px.bar(
    ranked, x='label', y='gap_score',
    color='coverage_ratio',
    color_continuous_scale='RdYlGn',
    range_color=[0, 1],
    title='Top 20 Most Overlooked Crises — Gap Score  (colour = funding coverage)',
    labels={'gap_score': 'Gap Score', 'label': '', 'coverage_ratio': 'Coverage'},
    text=ranked['coverage_ratio'].apply(lambda x: f'{x:.0%}' if pd.notna(x) else 'N/A')
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_tickangle=-45, yaxis=dict(title='Gap Score'))
fig.show()

In [ ]:
# Scatter: need vs gap score — labelled bubbles
fig = px.scatter(
    ranked,
    x='coverage_ratio', y='pin',
    size='gap_score', color='gap_score',
    text='country_iso3',
    log_y=True,
    color_continuous_scale='Reds',
    title='Coverage vs People in Need — bubble size = gap score',
    labels={'coverage_ratio': 'Coverage Ratio', 'pin': 'People in Need (log)'}
)
fig.update_traces(textposition='top center')
fig.add_vline(x=0.5, line_dash='dash', line_color='grey')
fig.update_layout(yaxis_tickformat='.2s')
fig.show()

---
## 4. World Map — Gap Scores

In [ ]:
# Use ALL scored in-scope rows (not just top 20) for the map
map_data = scored[scored['in_scope']].copy()

# Where multiple rows per country (sectors), take the ALL-sector row or max gap_score
map_agg = (
    map_data[map_data['sector'] == 'ALL']
    if 'ALL' in map_data['sector'].values
    else map_data.sort_values('gap_score', ascending=False).drop_duplicates('country_iso3')
)

fig = px.choropleth(
    map_agg,
    locations='country_iso3',
    color='gap_score',
    hover_name='country_iso3',
    hover_data={'pin': ':,.0f', 'coverage_ratio': ':.1%', 'gap_score': ':.3f'},
    color_continuous_scale='Reds',
    title='Humanitarian Gap Score by Country (red = most overlooked)',
    labels={'gap_score': 'Gap Score'}
)
fig.update_layout(geo=dict(showframe=False, showcoastlines=True, bgcolor='lightblue'))
fig.show()

In [ ]:
# Globe view — drag to rotate in the notebook
fig = px.choropleth(
    map_agg,
    locations='country_iso3',
    color='gap_score',
    hover_name='country_iso3',
    hover_data={'pin': ':,.0f', 'coverage_ratio': ':.1%'},
    color_continuous_scale='Reds',
    title='Humanitarian Gap Score — Interactive Globe  (drag to rotate)',
    labels={'gap_score': 'Gap Score'}
)
fig.update_geos(
    projection_type='orthographic',
    showland=True,     landcolor='#e8e8e8',
    showocean=True,    oceancolor='#a8d5e2',
    showlakes=True,    lakecolor='#a8d5e2',
    showrivers=False,
    showcoastlines=True, coastlinecolor='white',
    showcountries=True,  countrycolor='white',
    showframe=False
)
fig.update_layout(height=650)
fig.show()

---
## 5. geopandas Static Map

In [ ]:
# Natural Earth shapefile — cached to data/raw/ne_world.gpkg by scripts/download_data.py
_ne_path = Path('../data/raw/ne_world.gpkg')
if not _ne_path.exists():
    import geopandas as gpd
    world = gpd.read_file('https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip')
    world.to_file(_ne_path, driver='GPKG')
world = gpd.read_file(_ne_path)
world_gap = world.merge(
    map_agg[['country_iso3', 'gap_score', 'coverage_ratio']].rename(columns={'country_iso3': 'iso_a3'}),
    on='iso_a3', how='left'
)

fig, ax = plt.subplots(figsize=(18, 9))
world.plot(ax=ax, color='#eeeeee', edgecolor='white', linewidth=0.4)
world_gap.dropna(subset=['gap_score']).plot(
    ax=ax, column='gap_score', cmap='Reds',
    edgecolor='white', linewidth=0.4,
    legend=True, legend_kwds={'label': 'Gap Score', 'shrink': 0.5}
)
ax.set_title('Humanitarian Gap Score by Country', fontsize=15, pad=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('../docs/gap_score_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/gap_score_map.png')

In [ ]:
# Orthographic globe — centred on Africa/MENA where most crises are
try:
    world_gap_ortho = world_gap.to_crs('+proj=ortho +lat_0=5 +lon_0=25')
    world_ortho     = world.to_crs('+proj=ortho +lat_0=5 +lon_0=25')

    fig, ax = plt.subplots(figsize=(10, 10))
    world_ortho.plot(ax=ax, color='#eeeeee', edgecolor='white', linewidth=0.3)
    world_gap_ortho.dropna(subset=['gap_score']).plot(
        ax=ax, column='gap_score', cmap='Reds',
        edgecolor='white', linewidth=0.3,
        legend=True, legend_kwds={'label': 'Gap Score', 'shrink': 0.4, 'orientation': 'horizontal'}
    )
    ax.set_title('Humanitarian Gap Score — Globe View\n(Africa & MENA centred)', fontsize=13, pad=12)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../docs/gap_score_globe.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → docs/gap_score_globe.png')
except Exception as e:
    print(f'Globe projection failed: {e}\nUse the interactive Plotly globe above instead.')

---
## 6. Sector Breakdown for Top 5 Countries

In [ ]:
top5_countries = ranked['country_iso3'].head(5).tolist()

sector_breakdown = scored[
    scored['country_iso3'].isin(top5_countries) &
    scored['sector'].notna() &
    (scored['sector'] != 'ALL')
].copy()

if len(sector_breakdown) > 0:
    fig = px.bar(
        sector_breakdown.sort_values('pin', ascending=False),
        x='sector', y='pin', color='country_iso3', barmode='group',
        facet_col='country_iso3', facet_col_wrap=3,
        title='Sector-Level People in Need — Top 5 Most Overlooked Countries',
        labels={'pin': 'People in Need', 'sector': 'Sector'},
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False)
    fig.show()
else:
    print('No sector-level data for the top 5 countries in the current analysis window.')
    print('This is expected if only the 2026 HNO year is loaded (limited country coverage).')

---
## 7. LLM Query — Natural Language to Ranked Results

> Requires `ANTHROPIC_API_KEY` in `.env`

In [ ]:
from src.query.llm_parser import parse_query
from src.ranking.ranker import run_pipeline

QUERY = "Which crises have the highest proportion of people in need but the lowest fund allocations?"

filters = parse_query(QUERY)
print('Detected filters:')
print(f'  regions:            {filters.regions}')
print(f'  country_iso3s:      {filters.country_iso3s}')
print(f'  sectors:            {filters.sectors}')
print(f'  max_coverage_ratio: {filters.max_coverage_ratio}')
print(f'  min_pin:            {filters.min_pin}')
print(f'  year_range:         {filters.year_range}')
print(f'  chronic_only:       {filters.chronic_only}')

In [ ]:
llm_ranked = run_pipeline(master, filters, top_n=20)

llm_ranked[['rank', 'country_iso3', 'year', 'sector', 'pin',
            'requirements_usd', 'funding_usd', 'coverage_ratio', 'gap_score']]

In [ ]:
# Globe of LLM-filtered results
if len(llm_ranked) > 0:
    fig = px.choropleth(
        llm_ranked,
        locations='country_iso3',
        color='gap_score',
        hover_name='country_iso3',
        hover_data={'pin': ':,.0f', 'coverage_ratio': ':.1%'},
        color_continuous_scale='Reds',
        title=f'Query: "{QUERY}"',
    )
    fig.update_geos(
        projection_type='orthographic',
        showland=True, landcolor='#e8e8e8',
        showocean=True, oceancolor='#a8d5e2',
        showcoastlines=True, coastlinecolor='white',
        showcountries=True, countrycolor='white',
        showframe=False
    )
    fig.update_layout(height=600)
    fig.show()